<a href="https://colab.research.google.com/github/ConstantinaGio/aip-email-marketing_Group_32/blob/main/IB9BW0_Analytics_in_Practice_Group_32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IB9BW0 Analytics in Practice
## Group Assignment: E-Mail Marketing for Universal Plus
**Group:** 32

---

### Contents
1. Business Understanding
2. Data Understanding
3. Data Preparation

**Note:** This notebook runs from start to finish using only `campaign_data.csv` and `next_campaign.csv`, placed in the same folder as the notebook.

In [1]:
# upload campaign_data.csv and next_campaign.csv
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving next_campaign.csv to next_campaign (1).csv
Saving campaign_data.csv to campaign_data (1).csv
['next_campaign (1).csv', 'campaign_data (1).csv']


-----
## 1. Business Understanding

***Business problem:*** Universal Plus sends campaign e-mails mostly at random or by rules of thumb.Sending e-mails to uninterested customers costs money and several past campaigns were not successful.

***Analytics problem:*** predict whether each customer will visit the shop in the two weeks after a campaign, using only information available before the campaign. Because the answer is a category (yes or no), this is a *classification* problem.

**Key figures from the board:**
- Each e-mail costs **£3.50**.
- A visiting customer spends £93 on average, with a 30% margin, so each visit is worth about **£28**.



In [2]:
# Libraries we use in this notebook
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Figures given by the board
email_cost = 3.50
visit_value = 93 * 0.30          # average spend x margin
break_even = email_cost / visit_value

print(f"Value of a visit: £{visit_value:.2f}")
print(f"Break-even: {break_even:.1%}")

Value of a visit: £27.90
Break-even: 12.5%


-----
## 2. Data Understanding

### 2.1 Loading the data

In [4]:
# Read both files (only empty cells count as missing)
campaign = pd.read_csv("campaign_data.csv", keep_default_na=False, na_values=[""])
next_campaign = pd.read_csv("next_campaign.csv", keep_default_na=False, na_values=[""])

# Number of rows and columns
print("Campaign data:", campaign.shape)
print("Next campaign:", next_campaign.shape)

Campaign data: (52000, 21)
Next campaign: (12000, 20)


In [5]:
# The first five customers
campaign.head()

,Customer_ID,recency,purchase_segment,purchase,mens,womens,zip_area,new_customer,channel,email_segment,...,dependent,account,employed,phone,delivery,marriage,payment_card,satisfaction_score,spend,visit
0,30000001,12.0,1) 0 - 100,81.61,1,0,Urban,0,Phone,No E-Mail,...,1,1,1,1,1,0,1,NaN,0.0,0
1,30000004,NaN,3) 200 - 350,258.74,1,0,Urban,0,Multichannel,Mens E-Mail,...,1,1,1,1,3,0,1,8.0,0.0,0
2,30000005,6.0,3) 200 - 350,239.96,1,0,Suburban,0,Phone,No E-Mail,...,0,1,1,1,1,1,1,NaN,0.0,0
3,30000006,7.0,3) 200 - 350,268.18,1,1,Rural,1,Web,Womens E-Mail,...,1,1,1,1,3,2,0,5.0,0.0,0
4,30000007,1.0,3) 200 - 350,311.52,0,1,Suburban,0,Web,No E-Mail,...,0,1,1,1,3,1,1,8.0,0.0,0


In [6]:
# The first five customers in the next campaign file
next_campaign.head()

,Customer_ID,recency,purchase_segment,purchase,mens,womens,zip_area,new_customer,channel,email_segment,age,dependent,account,employed,phone,delivery,marriage,payment_card,satisfaction_score,spend
0,30000002,10.0,1) 0 - 100,97.84,0,1,Suburban,1,Phone,Womens E-Mail,42.0,1,1,1,1,3,1,1,7.0,0.0
1,30000003,12.0,1) 0 - 100,26.24,0,1,Urban,1,Phone,Mens E-Mail,38.0,1,1,1,1,1,0,1,NaN,0.0
2,30000020,2.0,3) 200 - 350,276.99,1,0,Suburban,0,Web,Womens E-Mail,23.0,0,1,1,1,1,2,1,7.0,0.0
3,30000027,8.0,3) 200 - 350,207.15,1,1,Rural,0,Phone,Womens E-Mail,27.0,0,1,1,1,1,1,1,NaN,0.0
4,30000030,5.0,1) 0 - 100,37.61,0,1,Urban,1,Web,No E-Mail,49.0,0,1,1,1,1,1,0,NaN,0.0


In [7]:
# The structure: column names, data types and the number of non-missing values
campaign.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52000 entries, 0 to 51999
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Customer_ID         52000 non-null  int64  
 1   recency             48238 non-null  float64
 2   purchase_segment    51977 non-null  object 
 3   purchase            52000 non-null  float64
 4   mens                52000 non-null  int64  
 5   womens              52000 non-null  int64  
 6   zip_area            51221 non-null  object 
 7   new_customer        52000 non-null  int64  
 8   channel             51241 non-null  object 
 9   email_segment       52000 non-null  object 
 10  age                 49987 non-null  float64
 11  dependent           52000 non-null  int64  
 12  account             52000 non-null  int64  
 13  employed            52000 non-null  int64  
 14  phone               52000 non-null  int64  
 15  delivery            52000 non-null  int64  
 16  marr

**Findings:**
- The campaign data contains 52,000 customers from the first campaign and 21 columns. The next campaign file contains 12,000 customers and 20 columns, as the `visit` column has been excluded.
- `Customer_ID` is a unique identifier and will therefore be removed before modelling.
- Four columns are stored as text (`purchase_segment`, `zip_area`, `channel`, `email_segment`), so they will need encoding.
- Some columns have fewer than 52,000 non-null values, so there are missing values to check and handle.

### 2.2 Missing values


In [8]:
# Count the missing values in each column and show them as a percentage,
# sorted so the column with the most missing values appears first.

# Number of missing values per column (isnull() and sum())
missing_count = campaign.isnull().sum()

# Share of missing values per column (%)
missing_percent = (campaign.isnull().sum() / len(campaign) * 100).round(1)

# Create a summary table with both number and percentage of missing values
missing_table = pd.DataFrame({"missing": missing_count, "percent": missing_percent})
missing_table[missing_table["missing"] > 0].sort_values("percent", ascending=False)

,missing,percent
satisfaction_score,31082,59.8
recency,3762,7.2
age,2013,3.9
channel,759,1.5
zip_area,779,1.5
spend,35,0.1
purchase_segment,23,0.0


In [9]:
# The same missing value check applied to the next campaign file (%)
(next_campaign.isnull().sum() / len(next_campaign) * 100).round(1)

,0
Customer_ID,0.0
recency,7.4
purchase_segment,0.1
purchase,0.0
mens,0.0
womens,0.0
zip_area,1.4
new_customer,0.0
channel,1.2
email_segment,0.0


**Findings:**
- `satisfaction_score` has the most missing values: about 60% in both files. This is because only customers who answered the survey have a score. *(readme.txt)*
- The other columns have a percentage of small missing values: `recency` (7.2%), `age` (3.9%), and less than 2% for `channel`, `zip_area`, `spend` and `purchase_segment`.
- If we drop every missing value, we would lose about 60% of the customers and introduce heavy bias into our model. So instead, we will fill the gaps.

Since both files follow the same pattern, they can be cleaned the same way


### 2.3 Data quality checks


In [10]:
# account: do all customers have an account? (1 = yes, 0 = no)
print(campaign["account"].value_counts(), "\n")

# employed: are almost all customers employed? (1 = yes, 0 = no)
print(campaign["employed"].value_counts(), "\n")

# spend: do only customers who visited spend money? (True = spent money)
print(pd.crosstab(campaign["spend"] > 0, campaign["visit"]), "\n")

# purchase_segment: do the price groups match the purchase amounts?
print(campaign.groupby("purchase_segment")["purchase"].agg(["min", "max"]), "\n")

# Are any values impossible? (min and max of the number columns)
print(campaign[["recency", "purchase", "age", "satisfaction_score"]].describe().round(1), "\n")

# Are there any duplicated customers?
print("Duplicate customers:", campaign["Customer_ID"].duplicated().sum())

account
1    52000
Name: count, dtype: int64 

employed
1    51914
0       86
Name: count, dtype: int64 

visit      0     1
spend             
False  43476    75
True       0  8449 

                      min      max
purchase_segment                  
1) 0 - 100          25.00    99.99
2) 100 - 200       100.00   199.99
3) 200 - 350       200.00   349.99
4) 350 - 500       350.04   499.99
5) 500 - 750       500.00   749.84
6) 750 - 1,000     750.16   999.93
7) 1,000 +        1000.04  3387.49 

       recency  purchase      age  satisfaction_score
count  48238.0   52000.0  49987.0             20918.0
mean       6.0     241.6     35.4                 7.0
std        3.5     256.7      8.9                 1.4
min        1.0      25.0     19.0                 2.0
25%        3.0      64.2     28.0                 6.0
50%        6.0     156.8     34.0                 7.0
75%        9.0     324.8     41.0                 8.0
max       12.0    3387.5     62.0                10.0 

Duplicate c

In [11]:
# employed: how many customers are not employed, and do they visit differently?
# employed: 1 = has a permanent job, 0 = does not

print(campaign.groupby("employed")["visit"].agg(["count", "mean"]).round(3), "\n")

          count   mean
employed              
0            86  0.093
1         51914  0.164 



**Findings:**
- `Customer_ID`: is just a label/identifier. Columns that only identify customers are not real information, so we remove it.
- `spend`: this is the total amount a customer spent in following two weeks after the campaign. Only customers who visited the shop spent money and we will know the amount spend after the campaign --> we cannot use it to decide who to e-mail. This is data leakage (Kapoor and Narayanan, 2023), so we remove it.
- `account`: every customer has an account,so this column gives the model no information, so we remove it.
- `employed`: Almost all customers are employed, with only 86 not employed. Non-employed customers visit the shop less often (9.3% compared with 16.4%). However, there are too few non-employed customers for the model to learn from, so it will be dropped.
- `purchase_segment`: it's the same as `purchase`, just in price groups, so it adds nothing new. We remove it.
- All values are realistic (e.g. age 19–62, recency 1–12 months) and there are no duplicate customers. Duplicates could put the same customer in both the training and test set, which is another type of leakage (Kapoor and Narayanan, 2023).


-----
## 3. Data Preparation

### 3.1 Removing columns
Based on the checks in 2.3, we remove the 5 columns that do not help the model or give away the answer.

In [12]:
# Columns to remove
drop_columns = ["Customer_ID", "spend", "account", "employed", "purchase_segment"]

# Remove them from both files, so they are the same
campaign = campaign.drop(columns=drop_columns)
next_campaign = next_campaign.drop(columns=drop_columns)

# Check the new size
print("Campaign data:", campaign.shape)
print("Next campaign:", next_campaign.shape)

Campaign data: (52000, 16)
Next campaign: (12000, 15)



### 3.2 Handling missing values

We do not delete rows, because we would lose about 60% of customers (see 2.2). Instead, we will fill those missing values:

| Column | Missing (%) | What we do | Why |
|---|---|---|---|
| `satisfaction_score` | 59.8 | Add a missing flag, then fill with the median | Only customers who answered the survey have a score, so a missing score tells us something |
| `recency` | 7.2 | Add a missing flag, then fill with the median | The median is the middle value, so a few very high or low numbers don't change it |
| `age` | 3.9 | Add a missing flag, then fill with the median | The median is the middle value, so a few very high or low numbers don't change it |
| `zip_area` | 1.5 | Fill with a new category "Missing"|Mark it as missing instead of guessing
| `channel` | 1.5 | Fill with a new category "Missing" | Mark it as missing instead of guessing |


The median is filled in after the train/test split, so it only comes from the training data.

In [13]:
# Add a missing flag for each number column (1 = was missing, 0 = not missing)
# As in Demo 1, we add the flags before filling, so we keep a record of the gaps
for column in ["recency", "age", "satisfaction_score"]:
    campaign[column + "_missing"] = campaign[column].isnull().astype(int)
    next_campaign[column + "_missing"] = next_campaign[column].isnull().astype(int)

In [14]:
# Do customers without a satisfaction score visit less often? (visit rate %)
# satisfaction_score_missing: 1 = no score, 0 = has a score
(campaign.groupby("satisfaction_score_missing")["visit"].mean() * 100).round(1)

,visit
satisfaction_score_missing,
0,20.4
1,13.7


In [15]:
# Fill the gaps in the text columns with a new category "Missing"
for column in ["zip_area", "channel"]:
    campaign[column] = campaign[column].fillna("Missing")
    next_campaign[column] = next_campaign[column].fillna("Missing")

# Check: which columns still have gaps?
print(campaign.isnull().sum()[campaign.isnull().sum() > 0])

recency                3762
age                    2013
satisfaction_score    31082
dtype: int64


**Findings:**
- We added three missing flags: `recency_missing`, `age_missing` and `satisfaction_score_missing`.
- Customers without a satisfaction score visit less often (13.7%) than customers with a score (20.4%). So whether a customer answered the survey is useful information.These will be filled with the median. The flag (1 = did not answer) helps the model understand which customers did not answer the survey.
- The gaps in `zip_area` and `channel` are now a "Missing" category.
- Only `recency`, `age` and `satisfaction_score` still have gaps. We fill these with the median after the train/test split.

### 3.3 Encoding
We need to turn all text types into binary (0/1) columns as models only work with numbers:
- `email_segment`: two new columns, `mens_email` and `womens_email` (1 = received it, 0 = did not). If both are 0, the customer got no e-mail.
- `zip_area`, `channel`, `marriage` and `delivery` have no natural order, so we use one-hot encoding.
-  `marriage` and `delivery` are stored as numbers, but they are groups (e.g. 1 = married, 2 = single), so they need to be encoded.

In [16]:
# E-mail received: two 0/1 columns, using np.where() as in Demo 2
# No e-mail = 0 in both columns
for data in [campaign, next_campaign]:
    data["mens_email"] = np.where(data["email_segment"] == "Mens E-Mail", 1, 0)
    data["womens_email"] = np.where(data["email_segment"] == "Womens E-Mail", 1, 0)

# Remove the original text column
campaign = campaign.drop(columns=["email_segment"])
next_campaign = next_campaign.drop(columns=["email_segment"])

In [20]:
# One 0/1 column per group (one-hot encoding, as in Demo 2)
encode_columns = ["zip_area", "channel", "marriage", "delivery"]

campaign = pd.get_dummies(campaign, columns=encode_columns, drop_first=True, dtype=int)
next_campaign = pd.get_dummies(next_campaign, columns=encode_columns, drop_first=True, dtype=int)

print("Campaign data:", campaign.shape)
print("Next campaign:", next_campaign.shape)

Campaign data: (52000, 26)
Next campaign: (12000, 25)


In [21]:
# Every column should now be a number
campaign.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52000 entries, 0 to 51999
Data columns (total 26 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   recency                     48238 non-null  float64
 1   purchase                    52000 non-null  float64
 2   mens                        52000 non-null  int64  
 3   womens                      52000 non-null  int64  
 4   new_customer                52000 non-null  int64  
 5   age                         49987 non-null  float64
 6   dependent                   52000 non-null  int64  
 7   phone                       52000 non-null  int64  
 8   payment_card                52000 non-null  int64  
 9   satisfaction_score          20918 non-null  float64
 10  visit                       52000 non-null  int64  
 11  recency_missing             52000 non-null  int64  
 12  age_missing                 52000 non-null  int64  
 13  satisfaction_score_missing  520

In [22]:
# Check 1: the new e-mail columns (1 = got this e-mail, both 0 = no e-mail)
print(campaign[["mens_email", "womens_email"]].head(), "\n")

# Check 2: list the 0/1 columns created by one-hot encoding
new_columns = [c for c in campaign.columns if c.startswith(("zip_area_", "channel_", "marriage_", "delivery_"))]
print("New 0/1 columns:", new_columns, "\n")

# Check 3: make sure there are no text columns left (only numbers)
print(campaign.dtypes.value_counts(), "\n")

# Check 4: both files should have the same columns (apart from visit)
print("Same columns:", list(campaign.drop(columns=["visit"]).columns) == list(next_campaign.columns))

   mens_email  womens_email
0           0             0
1           1             0
2           0             0
3           0             1
4           0             0 

New 0/1 columns: ['zip_area_Rural', 'zip_area_Suburban', 'zip_area_Urban', 'channel_Multichannel', 'channel_Phone', 'channel_Web', 'marriage_1', 'marriage_2', 'delivery_2', 'delivery_3'] 

int64      22
float64     4
Name: count, dtype: int64 

Same columns: True


**Findings:**
- The e-mail groups are now two 0/1 columns: `mens_email` and `womens_email`.
- `zip_area`, `channel`, `marriage` and `delivery` are now 0/1 columns, one for each group.
- All columns are now numbers, and both files have exactly the same columns.

Therefore, the data is ready to be split into train/test sets.



### 3.4 Train/test split and filling the remaining gaps

The data will be split into:
-  a training set (70%), used to build the models
-  a test set (30%), to check how well models predict customers.

We use `stratify=y` so both sets have the same share of visitors.

After the split, we fill the gaps in `recency`, `age` and `satisfaction_score` with the median. We only use the training set to work out the median, then use the same numbers for the test set and the next campaign.

Filling  the gaps before splitting, the test customers help calculate the median, so they influence the training data. If we split first and use only the training data, the test stays untouched. Filling gaps using the whole dataset would indroduce data leakage (Kapoor and Narayanan, 2023). This keeps the test set separate, so the test results show how the models would work on new customers



In [23]:
from sklearn.model_selection import train_test_split

# Features (everything except the target) and target (visit)
X = campaign.drop(columns=["visit"])
y = campaign["visit"]

# 70% training, 30% test (as in Demo 2)
# stratify=y keeps the share of visitors the same in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=10, stratify=y)

print("Training set:", X_train.shape[0], "customers")
print("Test set:    ", X_test.shape[0], "customers")

# Share of visitors in each set
print("\nAll data:\n", y.value_counts(normalize=True).round(3))
print("\nTraining set:\n", y_train.value_counts(normalize=True).round(3))
print("\nTest set:\n", y_test.value_counts(normalize=True).round(3))

Training set: 36400 customers
Test set:     15600 customers

All data:
 visit
0    0.836
1    0.164
Name: proportion, dtype: float64

Training set:
 visit
0    0.836
1    0.164
Name: proportion, dtype: float64

Test set:
 visit
0    0.836
1    0.164
Name: proportion, dtype: float64


In [24]:
# Median of each column, worked out from the training set only
fill_columns = ["recency", "age", "satisfaction_score"]
medians = X_train[fill_columns].median()
print("Medians from the training set:\n", medians)

# Use the same medians for all three datasets
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)
X_next = next_campaign.fillna(medians)

# Check: no gaps left anywhere
print("\nMissing values left:")
print("Training set:", X_train.isnull().sum().sum())
print("Test set:", X_test.isnull().sum().sum())
print("Next campaign:", X_next.isnull().sum().sum())

Medians from the training set:
 recency                6.0
age                   34.0
satisfaction_score     7.0
dtype: float64

Missing values left:
Training set: 0
Test set: 0
Next campaign: 0


In [27]:
# Check: are the training medians the same as the medians of all the data?
median_check = pd.DataFrame({
    "Training set": medians,
    "All data": X[fill_columns].median(),
})
median_check

,Training set,All data
recency,6.0,6.0
age,34.0,34.0
satisfaction_score,7.0,7.0


In [26]:
# How many values were filled in each dataset, and with which median?
# The missing flags (1 = was missing) tell us how many gaps were filled
filled_table = pd.DataFrame({
    "Filled with (median)": medians,
    "Filled in training set": [X_train[c + "_missing"].sum() for c in fill_columns],
    "Filled in test set": [X_test[c + "_missing"].sum() for c in fill_columns],
    "Filled in next campaign": [X_next[c + "_missing"].sum() for c in fill_columns],
})
filled_table

,Filled with (median),Filled in training set,Filled in test set,Filled in next campaign
recency,6.0,2626,1136,894
age,34.0,1412,601,505
satisfaction_score,7.0,21754,9328,7234


**Findings:**
- Training set: 36,400 customers. Test set: 15,600 customers.
- Both sets have the same share of visitors (16.4%).
- Since the medians from the training set are the same as for all the data (recency 6, age 34, satisfaction score 7). So filling after the split does not change the results, but it keeps the test set separate.



-----
### References
1. Kapoor, S. and Narayanan, A. (2023). Leakage and the reproducibility crisis in machine-learning-based science. Patterns, 4(9), pp.100804–100804. doi:10.1016/j.patter.2023.100804.